In [1]:
import logging
import tempfile
import zipfile
from pathlib import Path

import polars as pl
import polars.selectors as cs
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

# Only these years have the detailed (16-column) impo_*.lst report format;
# 2017-2018 zips carry an incompatible aggregated-report format instead.
START_YEAR = 2019
END_YEAR = 2026


def convert_month_to_parquet(zip_path: Path, interim_dir: Path) -> Path | None:
    """Convert one month's impo_*.lst report into its own parquet file.

    The report is extracted from its zip into a temporary directory, converted,
    and the temporary directory (and its extracted text) is discarded as soon as
    the parquet has been written, so at most one month's raw text (a few GB)
    ever sits on disk at a time.

    Args:
        zip_path: Path to a monthly AFIP zip archive (e.g. data/raw/2023/202301.zip).
        interim_dir: Root directory to write the monthly parquet under.

    Returns:
        Path to the written monthly parquet, or None if the zip has no impo_*.lst member.
    """
    year_month = zip_path.stem
    month_dir = interim_dir / year_month[:4] / year_month
    output_path = month_dir / f"impo_{year_month}.parquet"

    if output_path.exists():
        return output_path

    with zipfile.ZipFile(zip_path) as zf:
        impo_members = [
            name for name in zf.namelist() if Path(name).name.startswith("impo_")
        ]
        if not impo_members:
            logger.warning(f"No impo_*.lst found in {zip_path.name}, skipping.")
            return None

        month_dir.mkdir(parents=True, exist_ok=True)

        with tempfile.TemporaryDirectory() as tmp_dir:
            extracted_path = Path(zf.extract(impo_members[0], path=tmp_dir))

            # Eagerly read only the header (takes almost zero memory)
            header_df = pl.read_csv(
                extracted_path, separator="'", quote_char=None, n_rows=0
            )
            clean_columns = [
                col.strip().replace("\x0c", "") for col in header_df.columns
            ]

            # LAZY read: scan_csv doesn't load data into memory yet.
            # It just registers the instructions.
            lazy_df = pl.scan_csv(
                extracted_path,
                separator="'",
                quote_char=None,
                skip_rows=2,
                has_header=False,
                new_columns=clean_columns,
                infer_schema_length=0,
            )

            # Add cleaning instructions
            lazy_df = lazy_df.with_columns(cs.string().str.strip_chars())

            # The source .lst files are paginated reports: every form-feed page break
            # repeats the header row and a '---------' separator row, which otherwise
            # leak into the data as garbage rows. FECHA_ is always a 6-digit YYYYMM
            # value in real rows, so use it to drop those repeated header/separator rows.
            lazy_df = lazy_df.filter(pl.col("FECHA_").str.contains(r"^\d{6}$"))

            # Executes the instructions in streaming mode and writes the month's
            # parquet before the temporary directory (and extracted text) is cleaned up.
            lazy_df.sink_parquet(output_path)

    return output_path


def build_monthly_parquets(
    raw_dir: str = "../data/raw/", interim_dir: str = "../data/interim/"
) -> None:
    """Convert every in-range monthly zip under raw_dir into its own parquet file.

    Args:
        raw_dir: Directory containing data/raw/<year>/<yyyymm>.zip archives.
        interim_dir: Directory to write data/interim/<year>/<yyyymm>/impo_<yyyymm>.parquet files.
    """
    raw_path = Path(raw_dir)
    interim_path = Path(interim_dir)

    zip_files = [
        zip_path
        for zip_path in sorted(raw_path.rglob("*.zip"))
        if START_YEAR <= int(zip_path.stem[:4]) <= END_YEAR
    ]

    if not zip_files:
        logger.info(
            "No zip files found in the raw directory for the configured year range."
        )
        return

    logger.info(
        f"Converting {len(zip_files)} monthly zip files ({START_YEAR}-{END_YEAR})..."
    )

    pbar = tqdm(zip_files, desc="Converting months")
    for zip_path in pbar:
        pbar.set_postfix(current_file=zip_path.name)
        convert_month_to_parquet(zip_path, interim_path)


def combine_monthly_parquets(
    interim_dir: str = "../data/interim/", processed_dir: str = "../data/processed/"
) -> None:
    """Concatenate every monthly parquet in interim_dir into one combined parquet.

    Args:
        interim_dir: Directory containing data/interim/<year>/<yyyymm>/impo_<yyyymm>.parquet files.
        processed_dir: Directory to write the combined importaciones_<start>-<end>.parquet file.
    """
    interim_path = Path(interim_dir)
    out_path = Path(processed_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    monthly_parquets = sorted(interim_path.rglob("impo_*.parquet"))
    if not monthly_parquets:
        logger.info("No monthly parquet files found to combine.")
        return

    years_found = {int(p.stem.split("_")[1][:4]) for p in monthly_parquets}
    start_year, end_year = min(years_found), max(years_found)
    output_file_path = out_path / f"importaciones_{start_year}-{end_year}.parquet"

    logger.info(
        f"Combining {len(monthly_parquets)} monthly parquet files into {output_file_path}..."
    )

    final_lazy_df = pl.scan_parquet([str(p) for p in monthly_parquets])
    final_lazy_df.sink_parquet(output_file_path)

    logger.info("Done! You are ready to query.")


# Run the pipeline: convert each month's zip to its own parquet (skipping months
# already converted), then combine all monthly parquets into one processed file.
build_monthly_parquets()
combine_monthly_parquets()

Converting 92 monthly zip files (2019-2026)...


Converting months:   0%|          | 0/92 [00:00<?, ?it/s]

Combining 92 monthly parquet files into ..\data\processed\importaciones_2019-2026.parquet...
Done! You are ready to query.
